In [ ]:
# @title
!pip install -q "datasets==2.16.1" albumentations kaggle gradio
!pip install -q "transformers>=4.36" torch torchvision


In [ ]:
import os, gc, json, shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoProcessor, BlipForConditionalGeneration
from datasets import load_dataset
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
#upload the kaggle.json file

from google.colab import files
files.upload()

In [ ]:
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.copy("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
ZIP_PATH     = "/content/roco-dataset.zip"
EXTRACT_PATH = "/content/roco"

In [ ]:
def download_and_extract():
    # to download if the zip is missing
    if not os.path.exists(ZIP_PATH):
        print("Downloading ROCO dataset ...")
        os.system("kaggle datasets download -d virajbagal/roco-dataset -p /content")
    else:
        print("Zip already present — skipping download.")

    # unzip if the folder is missing
    if not os.path.exists(EXTRACT_PATH):
        print("Extracting ...")
        os.system(f"unzip -q {ZIP_PATH} -d {EXTRACT_PATH}")
    else:
        print("Already extracted — skipping unzip.")
    print("DONE")

download_and_extract()

In [ ]:
df_train = pd.read_csv(
    "/content/roco/all_data/train/radiologytraindata.csv", delimiter=","
)
print(f"Full set: {df_train.shape[0]} rows")

mask        = df_train["caption"].str.contains("chest x-ray", case=False, na=False)
filtered_df = df_train[mask].copy()                    # FIX 2: .copy()
print(f"Chest x-ray subset: {len(filtered_df)} rows")

IMAGE_SRC_DIR = "/content/roco/all_data/train/radiology/images/"
filtered_df["images"] = IMAGE_SRC_DIR + filtered_df["name"]

Full set: 65450 rows
Chest x-ray subset: 1735 rows


In [ ]:
WORK_DIR = "/content/working/train"
os.makedirs(WORK_DIR, exist_ok=True)

missing = []
for _, row in filtered_df.iterrows():
    src = row["images"]
    if os.path.exists(src):
        shutil.copy(src, WORK_DIR)
    else:
        missing.append(src)

if missing:
    print(f"Warning: {len(missing)} files not found — removing from df.")
    filtered_df = filtered_df[filtered_df["images"].apply(os.path.exists)].copy()

filtered_df.drop(columns=["images", "id"], inplace=True)

In [ ]:
captions = filtered_df.apply(
    lambda r: {"file_name": r["name"], "text": r["caption"]}, axis=1
).tolist()

jsonl_path = os.path.join(WORK_DIR, "metadata.jsonl")
with open(jsonl_path, "w") as f:
    for item in captions:
        f.write(json.dumps(item) + "\n")               # FIX 3: added \n

print(f"Wrote {len(captions)} records to {jsonl_path}")

Wrote 1735 records to /content/working/train/metadata.jsonl


In [ ]:
hf_dataset = load_dataset("imagefolder", data_dir=WORK_DIR, split="train")
print(hf_dataset)

In [ ]:
class ImageCaptioningDataset(Dataset):
    def __init__(self, dataset, processor):
        self.dataset   = dataset
        self.processor = processor

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item     = self.dataset[idx]
        encoding = self.processor(
            images=item["image"],
            text=item["text"],
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}
        encoding["labels"] = encoding["input_ids"].clone()  # FIX 4: labels required
        return encoding

In [ ]:
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "Salesforce/blip-image-captioning-base"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model     = BlipForConditionalGeneration.from_pretrained(MODEL_ID)
model.to(DEVICE)
print(f"Device: {DEVICE}")

In [ ]:
BATCH_SIZE  = 4
NUM_WORKERS = 2

train_dataset    = ImageCaptioningDataset(hf_dataset, processor)
train_dataloader = DataLoader(
    train_dataset, shuffle=True,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Batches per epoch: {len(train_dataloader)}")


Batches per epoch: 434


In [ ]:
NUM_EPOCHS  = 5
LR          = 5e-5
ACCUM_STEPS = 4           # effective batch = 16
SAVE_PATH   = "/content/blip-radiology-finetuned"

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS * len(train_dataloader)
)
scaler = GradScaler()

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(
        tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    ):
        input_ids      = batch["input_ids"].to(DEVICE)
        pixel_values   = batch["pixel_values"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)

        with autocast():
            outputs = model(
                input_ids=input_ids, pixel_values=pixel_values,
                attention_mask=attention_mask, labels=labels,
            )
            loss = outputs.loss / ACCUM_STEPS

        scaler.scale(loss).backward()

        if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_dataloader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUM_STEPS

    print(f"  Epoch {epoch+1}  avg loss: {total_loss / len(train_dataloader):.4f}")

In [ ]:
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
processor.save_pretrained(SAVE_PATH)
print(f"Saved to {SAVE_PATH}")

In [ ]:
from google.colab import drive; drive.mount("/content/drive")
shutil.copytree(SAVE_PATH,"/content/drive/MyDrive/blip-radiology-finetuned", dirs_exist_ok=True)



Mounted at /content/drive


'/content/drive/MyDrive/blip-radiology-finetuned'

In [ ]:
def generate_caption(image, mdl, proc, device, max_new_tokens=100):
    mdl.eval()
    inputs = proc(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = mdl.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=4)
    return proc.decode(ids[0], skip_special_tokens=True)

print("Sample:", generate_caption(hf_dataset[0]["image"], model, processor, DEVICE))

Sample: chest x - ray showing right - sided pneumothorax.


In [ ]:
import gradio as gr

infer_processor = AutoProcessor.from_pretrained(SAVE_PATH)
infer_model     = BlipForConditionalGeneration.from_pretrained(SAVE_PATH).to(DEVICE)
infer_model.eval()

def gradio_caption(pil_image):
    if pil_image is None:
        return "Please upload an image."
    return generate_caption(pil_image, infer_model, infer_processor, DEVICE)

demo = gr.Interface(
    fn=gradio_caption,
    inputs=gr.Image(type="pil", label="Upload Radiology Image"),
    outputs=gr.Textbox(label="Generated Caption", lines=4),
    title="BLIP Radiology Caption Generator",
    description="Fine-tuned on ROCO (chest x-ray subset). Upload a radiology image.",
    allow_flagging="never",
)
demo.launch()
